# Compcor Comparison
- Metrics and setup taken from [the compcor library](https://github.com/IBM/comparing-corpora) and the [meme setup](https://github.com/IBM/meme) from ["Measuring the Measuring Tools" by Kour et al.](https://doi.org/10.18653/v1/2022.gem-1.35)

In [ ]:
import time
import random
import torch
import os
import glob
import sklearn

import pandas as pd
import polars as pl
import numpy as np


import scipy
import statsmodels.api as sm
from statsmodels.formula.api import ols


import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder
from KSC import KSC

from itertools import combinations

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]
ksc_measures = ['Accuracy', 'Weighted Accuracy', 'Time', 'Monotonicity', 'Separability', 'Linearity']
# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

In [ ]:
# ------------------ Loading and summarizing data functions. (utils.py) ------------------

# Helper function to load a corpus.
def load_corpus(corpus_path, max_samples=np.inf):
	file = open(corpus_path, 'r', encoding='utf-8')
	sentences = [sentence for sentence in [line.strip() for line in file.readlines()] if sentence]
	return sentences

# Helper function to load a labelled corpus.
def load_labeled_corpus(filename, sep='\t', max_samples=np.inf):
	data = pd.read_csv(filename, sep=sep, names=["label", "sample"], skipinitialspace=True, index_col=False)
	data.drop(np.where(pd.isnull(data))[0], axis=0, inplace=True)
	data = data.apply(lambda x: x.str.strip())
	data = data.sample(frac=1, random_state=1).reset_index()
	if not np.isinf(max_samples):
		data = data.head(max_samples)
	data.sort_values("label", ascending=True, inplace=True)
	sentences = data['sample'].to_numpy()
	labels = data['label'].to_numpy()
	return sentences, labels

# Helper function to load data.
def load_data(max_samples=np.inf):
	clinc, _ = load_labeled_corpus('data/clinc150_uci_data.tsv', max_samples=max_samples)
	banking, _ = load_labeled_corpus('data/banking77_data.tsv', max_samples=max_samples)
	atis, _ = load_labeled_corpus('data/atis_intents.csv', sep=',',max_samples=max_samples)
	yahoo, _ = load_labeled_corpus('data/yahoo_data.tsv', max_samples=max_samples)

	return clinc, banking, atis, yahoo

# Helper function to summarize results.
def summarize_results(metrics_measures_df):
	mu = metrics_measures_df.groupby(['metric']).mean()
	mu = mu.round(decimals=3)
	std = metrics_measures_df.groupby(['metric']).std()
	return mu, std

In [ ]:
# ------------------ Functions to compute metric characteristics. (metric_characteristics.py) ------------------

# Helper function for metric monotonicity.
def metric_monotonicity(ells, distances):
	return scipy.stats.spearmanr(ells, distances).correlation

# Helper function for metric separability.
def metric_separability(ells, distances):
	df = pd.DataFrame(data=list(zip(ells, distances)), columns=['ell', 'distance'])
	model = ols('distance ~ C(ell)', data=df).fit()
	aov_table = sm.stats.anova_lm(model, typ=2)
	return anova_table(aov_table)['omega_sq'][0]

# Helper function for anova table.
def anova_table(aov):
	aov['mean_sq'] = aov[:]['sum_sq'] / aov[:]['df']
	aov['eta_sq'] = aov[:-1]['sum_sq'] / sum(aov['sum_sq'])
	aov['omega_sq'] = (aov[:-1]['sum_sq'] - (aov[:-1]['df'] * aov['mean_sq'][-1])) / (
				sum(aov['sum_sq']) + aov['mean_sq'][-1])
	cols = ['sum_sq', 'df', 'mean_sq', 'F', 'PR(>F)', 'eta_sq', 'omega_sq']
	aov = aov[cols]
	return aov

# Helper function for metric linearity.
def metric_linearity(ells, distances):
	return scipy.stats.linregress(ells, y=distances).rvalue

# Helper function for robustness.
def metric_size_robustness(sizes, distances, true_distance):
	return 1 - np.nansum(np.abs((distances - true_distance))) / (true_distance * 10 * len(np.unique(sizes)))

# Helper function for metric imbalance robustness.
def metric_imbalance_robustness(sizes, comp_sizes, distances, true_distance):
	return 1 - sum(np.abs((distances - true_distance))) / (10 * len(np.unique(sizes)))

In [ ]:
'''

# ------------------ Functions to compute Increasingly Fine-tuned Corpora experiments. (IFC_experiment.py) ------------------

SMALL_SIZE = 25
matplotlib.rc('font', size=SMALL_SIZE)
matplotlib.rc('axes', titlesize=SMALL_SIZE)

# Compute Increasingly Fine-tuned Corpora metrics.
def compute_IFC(metrics, metrics_names, reference_corpus, generated_corpora_path, rang,
                output_path, num_samples, repetitions=10):

    original, _ = load_labeled_corpus(reference_corpus, max_samples=3000)

    distance_trend = []
    self_distance = []
    for i in rang:
        #sentences = utils.load_corpus(os.path.join(generated_corpora_path,'iteration_{}.txt'.format(i)), max_samples=3000)
        generated, _ = load_labeled_corpus(os.path.join(generated_corpora_path ,'{}.tsv').format(i), max_samples=3000)
        for m,metric in enumerate(metrics):
            print(metric)
            generated_c = np.array(get_metric_dependant_data(metric, generated))
            original_c = np.array(get_metric_dependant_data(metric, original))
            for rep in range(repetitions):
                indices = random.sample(range(len(original_c)), num_samples)
                indices_generated = random.sample(range(len(generated_c)), num_samples)
                original_subset = original_c[indices].tolist()
                generated_subset = generated_c[indices_generated].tolist()
                distance_trend.append([i, metrics_names[m],rep, metric(original_subset,generated_subset)])

            #Estimate reference-reference distance
            indices = random.sample(range(len(original_c)), num_samples)
            original_subset1 = original_c[indices].tolist()
            indices = random.sample(range(len(original_c)), num_samples)
            original_subset2 = original_c[indices].tolist()

            self_distance.append([i, metrics_names[m], metric(original_subset1, original_subset2)])

    df_IFC_distance = pd.DataFrame(data=distance_trend, columns=['iteration', 'metric', 'rep', 'distance'])
    df_reference_self_distance = pd.DataFrame(data=self_distance, columns=['iteration', 'metric', 'distance'])

    df_IFC_distance.to_csv(path_or_buf=os.path.join(output_path, 'ifc_distance.csv'), index=False)
    df_reference_self_distance.to_csv(path_or_buf=os.path.join(output_path, 'df_reference_self_distance.csv'), index=False)

    return df_IFC_distance, df_reference_self_distance

# Plot Increasingly Fine-tuned Corpora metrics.
def plot_IFC(df_distance, df_self_distance):
    plt.style.use('seaborn-whitegrid')
    metrics_names = np.unique(df_distance['metric'])

    fig, ax = plt.subplots(1, len(metrics_names), figsize=(35, 5))
    for i, metric_name in enumerate(metrics_names):
        metric_df = df_distance[df_distance['metric'] == metric_name]

        sns.scatterplot(x='iteration', y='distance', data=metric_df, ax=ax[i], color='orange',  s=40)
        sns.regplot(x='iteration', y='distance', data=metric_df, ax=ax[i],
                    scatter=False, truncate=False, color='blue')
        self_metric_df = df_self_distance[df_self_distance['metric'] == metric_name]

        x_min_max = [np.min(self_metric_df['iteration']), np.max(self_metric_df['iteration'])]
        mean_self_distance = np.mean(self_metric_df['distance'])
        sns.lineplot(x=x_min_max, y=mean_self_distance, ax=ax[i], color='green', linewidth=2)

    [axi.set_title(metrics_names[i]) for i, axi in enumerate(ax)]
    [axi.set(xlabel=None) for axi in ax]
    [axi.set(ylabel=None) for axi in ax]

    plt.subplots_adjust(left=0.05,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

    plt.show()

# ISABEL HERE - PATHS NEED TO BE CHANGED IF USING BUT YOU HAVE NOT GENERATED DATA FOR THIS YET
reference_path = 'data/banking77_data.tsv' # or './data/news_data/news_data.tsv'
generated_path = 'data/banking77_generated' # or './data/news_data/generated_news_data/'
sampled_samples = 700 # or sampled_samples = 100
rang = range(1, 35, 1) # or rang = range(1, 75, 2)


df_distance, df_self_distance = compute_IFC(metrics,metrics_names,reference_path, generated_path,
                                            rang=rang,
                                            output_path=os.path.join('/outputCompcor/IFC','banking_77_new'),
                                            num_samples=sampled_samples)

plot_IFC(df_distance, df_self_distance)

'''

In [14]:
def make_input_data(file_paths="./outputsTrain/*/texts_and_ids.csv"):
    # Get all CSV files.
    all_files = glob.glob(file_paths)  # change this to your folder path

    # Loop through files.
    all_dfs = {}

    for f in all_files:
        df = pd.read_csv(f)
        df['category'] = df['doc_id'].apply(lambda x: x.split('_')[0])
        all_dfs[f.split('/')[-2]] = df

    # print(all_dfs)

    # Concatenate all aggregated DataFrames
    # dfs = pd.concat(all_dfs, ignore_index=True)
    # dfs = dfs.drop_duplicates(subset='doc_id')
    # Return df
    return all_dfs

train_dfs = make_input_data()
# dataset_names = train_df['category'].value_counts().index.values.tolist()
# datasets_dict = {}
# for dataset_name in dataset_names:
#     print(f"{dataset_name}: {len(train_df[train_df['category'] == dataset_name])}")
#     datasets_dict[dataset_name] = train_df[train_df['category'] == dataset_name]['text'].values.to_numpy()

train_dfs

{'1618':                                                  text       doc_id category
 0   what are the flights from atlanta to baltimore...     atis_988     atis
 1   show me the flights from boston to fort worth ...    atis_1448     atis
 2   i want a flight from montreal quebec to san di...    atis_1577     atis
 3   i would like a flight from washington to bosto...    atis_2325     atis
 4   what are all flights from pittsburgh to boston...    atis_2129     atis
 ..                                                ...          ...      ...
 95  You have an upper respiratory infection which ...  yahoo_15636    yahoo
 96  It's a loaded phrase that advertisers use to g...  yahoo_39692    yahoo
 97  Electromagnetic is a term that refers to elect...   yahoo_2237    yahoo
 98  Electrolysis. Running a current through water ...  yahoo_52316    yahoo
 99  Matthew....drink water, water, water...it help...   yahoo_7989    yahoo
 
 [100 rows x 3 columns],
 '1619':                                 

In [ ]:
for (name1, df1), (name2, df2) in combinations(train_dfs.items(), 2):

In [ ]:
# ------------------ Functions to compute Known Similarity Corpora experiments. (KSC_experiment.py) ------------------

SMALL_SIZE = 15
matplotlib.rc('font', size=SMALL_SIZE)
matplotlib.rc('axes', titlesize=SMALL_SIZE)

sns.set_theme(style="whitegrid", font_scale=2)

def runKSC(metrics, metric_names, corpus1:Corpus, corpus2:Corpus, n=30, k=7, repetitions=5, output="ksc"):
	ksc_results = []
	distance_results = []

	for metric_idx, metric in enumerate(metrics):
		c1 = get_metric_dependant_data(metric, corpus1)
		c2 = get_metric_dependant_data(metric, corpus2)

		for rep in range(repetitions):

			distances_metric = []

			ksc = KSC._known_similarity_corpora(c1, c2, n=n, k=k, unique_samples_corpora=True)
			start = time.time()
			accuracy, weighted_accuracy, distance_stats = KSC.test_ksc(ksc, dist=metric)
			ksc_time = (time.time() - start) / len(distance_stats)

			distances_metric.append(
				np.vstack([[metric_names[metric_idx], rep, a, b, b - a, y] for (a, b, y) in distance_stats]))

			distances_metric = np.vstack(distances_metric)

			# normalize the score for a specific metric.
			distances_metric = np.append(distances_metric, sklearn.preprocessing.StandardScaler().fit_transform(
				distances_metric[:, 5].reshape(-1, 1)), axis=1)
			distance_results.extend(distances_metric)

			ells = distances_metric[:, 4].astype('float')
			ds_normalized = distances_metric[:, 6].astype('float')

			monotonicity = metric_monotonicity(ells, ds_normalized)
			separability = metric_separability(ells, ds_normalized)
			linearity = metric_linearity(ells, ds_normalized)
			ksc_results.append(
				[metric_names[metric_idx], accuracy, weighted_accuracy, ksc_time, monotonicity, separability, linearity])

	metrics_measures_df = pd.DataFrame(data=ksc_results, columns=['metric'] + ksc_measures)
	metrics_measures_df['Time'] = (1 / metrics_measures_df['Time'])/100

	all_distance_samples_df = pd.DataFrame(data=distance_results,
									columns=['metric', 'repetition', 'i', 'j', 'l', 'distance', 'distance_score'])
	all_distance_samples_df["l"] = pd.to_numeric(all_distance_samples_df["l"])
	all_distance_samples_df["distance"] = pd.to_numeric(all_distance_samples_df["distance"])
	all_distance_samples_df["distance_score"] = pd.to_numeric(all_distance_samples_df["distance_score"])
	metrics_measures_df.to_csv(path_or_buf=output+'metrics_measures_df.csv', index=False, float_format='%.3f')
	all_distance_samples_df.to_csv(path_or_buf=output+'all_distance_samples_df.csv', index=False, float_format='%.3f')

	return metrics_measures_df, all_distance_samples_df


def plotKSC(all_distance_samples_df):

	sns.set_theme(style="whitegrid", font_scale=2)

	metrics_names = np.unique(all_distance_samples_df['metric'])
	fig, axlist = plt.subplots(1, len(metrics_names), figsize=(35, 5))
	for i, metric in enumerate(metrics_names):
		metric_df = all_distance_samples_df[all_distance_samples_df['metric'] == metric]
		sns.scatterplot(x='l', y='distance', data=metric_df, ax=axlist[i], color='orange')
		sns.regplot(x='l', y='distance', data=metric_df, ax=axlist[i],
					scatter=False, truncate=False)
		axlist[i].set_title('{}'.format(metric))
	[axi.set(xlabel=None) for axi in axlist]
	[axi.set(ylabel=None) for axi in axlist]

	plt.subplots_adjust(left=0.05,
						bottom=0.1,
						right=0.99,
						top=0.9,
						wspace=0.3,
						hspace=0.4)

	plt.show()


def plot_measures_results(metrics_measures_df):
	fig, ax = plt.subplots(1, 6, figsize=(35, 5))
	[sns.boxplot(ax=ax[i], x='metric', y=measure, data=metrics_measures_df) for i, measure in enumerate(ksc_measures)]

	plt.subplots_adjust(left=0.1,
						bottom=0.1,
						right=0.99,
						top=0.9,
						wspace=0.3,
						hspace=0.4)

	[axi.set(xlabel=None) for axi in ax]
	[axi.set_xticklabels(ax[0].get_xticklabels(), fontsize=12) for axi in ax]
	# plt.savefig('ksc_metrics.png')
	plt.show()


L = [20, 5]
H = [10, 110]
rep = 5

max_samples = H[0] * H[1] * rep
output_folder = 'outputCompcor/ksc_measures'
# ISABEL HERE MAKE THIS WORK BETTER!
# clinc, banking, atis, yahoo = load_data(max_samples)

for R in [L,H]:
	for i, pair in enumerate([('clinc150', 'banking77'), ('atis', 'yahoo')]):
		results_file_name = os.path.join(output_folder, f"pair{i}_{R[0]}_{R[1]}_")
		metrics_measures_df, all_distance_samples_df = runKSC(metrics, metrics_names, datasets_dict[pair[0]], datasets_dict[pair[1]], n=R[0], k=R[1], repetitions=rep, output=results_file_name)
		plotKSC(all_distance_samples_df)
		plot_measures_results(metrics_measures_df)
		mu12, std12 = summarize_results(metrics_measures_df)

In [ ]:
# ------------------ Functions to compute size imbalance experiments. (size_imbalance_experiment.py) ------------------

SMALL_SIZE = 15
matplotlib.rc('font', size=SMALL_SIZE)
matplotlib.rc('axes', titlesize=SMALL_SIZE)

sns.set_theme(style="whitegrid", font_scale=2)


def size_imbalance_sensitivity_experiment(metrics, metrics_names, corpus1, corpus2, sizes, repetitions, output_folder):
	distance_results = []
	source_corpora_distance = []
	for metric_idx, metric in enumerate(metrics):
		metric_distances = []
		for rep in range(repetitions):
			c1 = get_metric_dependant_data(metric, corpus1)
			c2 = get_metric_dependant_data(metric, corpus2)
			c1 = np.array(c1)
			c2 = np.array(c2)

			for (s, sc) in zip(sizes, reversed(sizes)):
				indices = random.sample(range(len(c1)), s)
				set1 = list(c1[indices])

				indices = random.sample(range(len(c2)), sc)
				set2 = list(c2[indices])

				indices = random.sample(range(len(c2)), s)
				set2_same_size = list(c2[indices])

				dist_complemeting = metric(set1, set2)
				dist_same_size = metric(set1, set2_same_size)

				metric_distances += [[metrics_names[metric_idx], rep, s, sc, dist_same_size, dist_complemeting]]

		metric_distances = np.stack(metric_distances)

		normalizer = sklearn.preprocessing.StandardScaler().fit(metric_distances[:, 4].reshape(-1, 1))

		metric_distances = np.append(metric_distances, normalizer.transform(metric_distances[:, 4].reshape(-1, 1)),
									 axis=1)
		metric_distances = np.append(metric_distances, sklearn.preprocessing.StandardScaler().fit_transform(
			metric_distances[:, 5].reshape(-1, 1)), axis=1)

		metric_distances[:, range(1, 7)] = metric_distances[:, range(1, 7)].astype(float)
		distance_results += metric_distances.tolist()
		sources_distance = metric(c1, c2)
		source_corpora_distance += [[metrics_names[metric_idx], sources_distance,
									 normalizer.transform((sources_distance).reshape(-1, 1))[0][0]]]

	size_imbalance_df = pd.DataFrame(data=distance_results,
									 columns=['metric', 'repetition', 'size', 'size_complementing', 'distance(same)',
											  'distance(comp)', 'distance(same)_norm', 'distance(comp)_norm'])
	source_corpora_distance_df = pd.DataFrame(data=source_corpora_distance,
											  columns=['metric', 'distance', 'distance_norm'])

	size_imbalance_df.to_csv(
		path_or_buf=os.path.join(output_folder, corpus1.name + corpus2.name + 'size_imbalance_df.csv'))
	source_corpora_distance_df.to_csv(
		path_or_buf=os.path.join(output_folder, corpus1.name + corpus2.name + 'source_corpora_distance_df.csv'))

	return size_imbalance_df, source_corpora_distance_df


def size_imbalance_robustness_measure(size_imbalance_df, source_corpora_distance_df):
	source_corpora_distance_df['size_robustness'] = np.empty(len(source_corpora_distance_df))
	source_corpora_distance_df['imbalance_robustness'] = np.empty(len(source_corpora_distance_df))
	for metric_name in np.unique(size_imbalance_df['metric']):
		metric_sizes_distance_samples = size_imbalance_df[size_imbalance_df['metric'] == metric_name]
		metric_true_sources_distance = \
		source_corpora_distance_df[source_corpora_distance_df['metric'] == metric_name]['distance'].iloc[0]

		metric_size_sens = metric_size_robustness(list(metric_sizes_distance_samples['size']),
												  list(metric_sizes_distance_samples['distance(same)']),
												  metric_true_sources_distance)

		metric_imbalance_sens = metric_imbalance_robustness(list(metric_sizes_distance_samples['size']),
															list(metric_sizes_distance_samples['size_complementing']),
															metric_sizes_distance_samples['distance(comp)'],
															metric_true_sources_distance)

		source_corpora_distance_df.loc[
			source_corpora_distance_df['metric'] == metric_name, 'size_robustness'] = metric_size_sens
		source_corpora_distance_df.loc[
			source_corpora_distance_df['metric'] == metric_name, 'imbalance_robustness'] = metric_imbalance_sens

	return size_imbalance_df, source_corpora_distance_df


# columns = 'distance(comp)_norm' or 'distance(same)_norm'
def plot_size_imbalance_scatter(df_distances, df_distance_corpora_all, column='distance(same)_norm'):
	# Save a palette to a variable:
	palette = sns.color_palette("Paired")
	metrics_names = np.unique(df_distances['metric'])
	x_min_max = [np.min(df_distances['size']), np.max(df_distances['size'])]
	fig, ax = plt.subplots(1, len(metrics_names), figsize=(len(metrics_names) * 5, 5))
	for i, metric in enumerate(metrics_names):
		sns.scatterplot(x='size', y=column, data=df_distances[df_distances['metric'] == metric],
						ax=ax[i], color=palette[1], s=50)
		df_metric = df_distance_corpora_all[df_distance_corpora_all['metric'] == metric]
		sns.lineplot(x_min_max, [np.mean(df_metric['distance']), np.mean(df_metric['distance'])], ax=ax[i],
					 linewidth=3, color=palette[2])
		ax[i].set_title('{}'.format(metric))

	[axi.set(xlabel=None) for axi in ax]
	[axi.set(ylabel=None) for axi in ax]

	plt.subplots_adjust(left=0.05,
						bottom=0.1,
						right=0.99,
						top=0.9,
						wspace=0.3,
						hspace=0.4)

	# plt.savefig('size_sens.png')
	plt.show()


# N = 2900
# repetitions = 10
# start = 50
# step = 200

# output_folder = 'outputCompcor/size_robustness'
# # ISABEL MAKE THIS WORK BETTER!
# clinc, banking, atis, yahoo = load_data()
# for i, pair in enumerate([(clinc, banking), (atis, yahoo)]):

# 	size_imbalance_df, source_corpora_distance_df = size_imbalance_sensitivity_experiment(metrics, metrics_names, pair[0], pair[1],
# 																							list(range(start, N+ 1, step)), repetitions, output_folder)

# 	# size_imbalance_df = pd.read_csv(filepath_or_buffer='src/experiments/meme/output/size_robustness/atis_yahoo_size_imbalance_df.csv')
# 	# source_corpora_distance_df = pd.read_csv(
# 	# 	filepath_or_buffer='src/experiments/meme/output/size_robustness/atis_yahoo_source_corpora_distance_df.csv')

# 	source_corpora_distance_df = source_corpora_distance_df.sort_values(by="metric", ascending=1)

# 	size_imbalance_df, source_corpora_distance_df = size_imbalance_robustness_measure(size_imbalance_df, source_corpora_distance_df)
# 	plot_size_imbalance_scatter(size_imbalance_df, source_corpora_distance_df, column='distance(same)')
# 	plot_size_imbalance_scatter(size_imbalance_df, source_corpora_distance_df,column='distance(comp)')